# Fashion MNIST — Tuned Mini-ResNet Demo

**Course:** Neural Networks — Final Project  
**Institution:** Instituto Tecnológico y de Estudios Superiores de Monterrey  
**Team 2:** Cesar Castaño · Félix Martínez · Raúl Valdés · Diego Marquez

---

This notebook loads our best-performing model (Tuned Mini-ResNet) directly from the project repository and lets you run inference on any sample from the Fashion MNIST test set.

### What you can do
- **Pick a specific sample** — set `USE_RANDOM = False` and choose a `SAMPLE_INDEX` (0–9 999)
- **Pick a random sample** — set `USE_RANDOM = True`

### Runtime requirement
> **GPU recommended.** Go to `Runtime → Change runtime type → T4 GPU` for faster loading.

---

## 1 — Environment Setup

Install dependencies and clone the project repository.

In [ ]:
# Install required packages
!pip install -q kagglehub tensorflow

In [ ]:
import os
import sys

REPO_URL  = 'https://github.com/CesarEmilioC/fashionMNIST_CNN_team2.git'
REPO_ROOT = '/content/fashionMNIST_CNN_team2'  # fixed Colab absolute path

if not os.path.exists(REPO_ROOT):
    !git clone {REPO_URL} {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

print(f'Repository ready at: {REPO_ROOT}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 2 — Load Fashion MNIST Test Data

In [ ]:
from src.data.loader import CLASS_NAMES, load_fashion_mnist_from_kaggle, preprocess_data

train_df, test_df = load_fashion_mnist_from_kaggle()
_, _, X_test, _, _, y_test = preprocess_data(train_df, test_df, seed=42)

print(f'Test set: {X_test.shape[0]} samples | Classes: {len(CLASS_NAMES)}')

## 3 — Load the Tuned Mini-ResNet

The model is loaded from `models/saved/tuned_mini_resnet.h5` inside the cloned repository.

In [ ]:
MODEL_PATH = os.path.join(REPO_ROOT, 'models', 'saved', 'tuned_mini_resnet.h5')

model = tf.keras.models.load_model(MODEL_PATH)
print(f'Model loaded from: {MODEL_PATH}')
model.summary()

---
## 4 — Choose Your Sample

**Edit the two variables below, then run this cell and the next:**

| Variable | Options | Description |
|---|---|---|
| `USE_RANDOM` | `True` / `False` | If `True`, a random test sample is selected |
| `SAMPLE_INDEX` | `0` – `9999` | Used only when `USE_RANDOM = False` |

In [ ]:
# ─────────────────────────────────────────────
#  EDIT THESE TWO LINES
USE_RANDOM   = True    # True → random sample | False → use SAMPLE_INDEX
SAMPLE_INDEX = 0       # integer 0–9999 (ignored when USE_RANDOM=True)
# ─────────────────────────────────────────────

if USE_RANDOM:
    idx = np.random.randint(0, len(X_test))
    print(f'Random sample selected — index {idx}')
else:
    if not (0 <= SAMPLE_INDEX < len(X_test)):
        raise ValueError(f'SAMPLE_INDEX must be between 0 and {len(X_test)-1}')
    idx = SAMPLE_INDEX
    print(f'Using fixed sample — index {idx}')

sample_image = X_test[idx]          # shape (28, 28, 1)
true_label   = y_test[idx]

## 5 — Run Inference & Visualize Results

In [ ]:
# Predict
probs      = model.predict(sample_image[np.newaxis, ...], verbose=0)[0]  # (10,)
pred_class = int(np.argmax(probs))
confidence = float(probs[pred_class])

is_correct = pred_class == true_label
result_color = '#27ae60' if is_correct else '#e74c3c'
result_label = 'CORRECT ✓' if is_correct else 'INCORRECT ✗'

# ── Figure layout ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
gs  = fig.add_gridspec(1, 2, width_ratios=[1, 2], wspace=0.35)

# Left panel — sample image
ax_img = fig.add_subplot(gs[0])
ax_img.imshow(sample_image.squeeze(), cmap='gray')
ax_img.set_title(
    f'Sample #{idx}\nTrue label: {CLASS_NAMES[true_label]}',
    fontsize=13, pad=10
)
ax_img.axis('off')

# Colored border to signal correct / incorrect
for spine in ax_img.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor(result_color)
    spine.set_linewidth(4)

# Right panel — confidence bar chart
ax_bar = fig.add_subplot(gs[1])
colors = ['#3498db'] * 10
colors[pred_class] = result_color

bars = ax_bar.barh(CLASS_NAMES, probs, color=colors, edgecolor='white', linewidth=0.5)
ax_bar.set_xlim(0, 1.08)
ax_bar.set_xlabel('Confidence', fontsize=12)
ax_bar.set_title('Class Probability Distribution', fontsize=13)
ax_bar.invert_yaxis()
ax_bar.grid(True, axis='x', alpha=0.3)

for bar, prob in zip(bars, probs):
    if prob > 0.005:
        ax_bar.text(
            bar.get_width() + 0.01,
            bar.get_y() + bar.get_height() / 2.,
            f'{prob:.1%}', va='center', fontsize=9
        )

# Legend
correct_patch   = mpatches.Patch(color='#27ae60', label='Predicted (correct)')
incorrect_patch = mpatches.Patch(color='#e74c3c', label='Predicted (incorrect)')
default_patch   = mpatches.Patch(color='#3498db', label='Other classes')
ax_bar.legend(handles=[correct_patch, incorrect_patch, default_patch],
              loc='lower right', fontsize=9)

# Main title
fig.suptitle(
    f'Prediction: {CLASS_NAMES[pred_class]}   |   Confidence: {confidence:.1%}   |   {result_label}',
    fontsize=14, fontweight='bold', color=result_color, y=1.02
)

plt.tight_layout()
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────
print('─' * 50)
print(f'Sample index   : {idx}')
print(f'True label     : {CLASS_NAMES[true_label]} (class {true_label})')
print(f'Predicted label: {CLASS_NAMES[pred_class]} (class {pred_class})')
print(f'Confidence     : {confidence:.4f} ({confidence:.1%})')
print(f'Result         : {result_label}')
print('─' * 50)
print('\nFull probability distribution:')
for i, (cls, p) in enumerate(zip(CLASS_NAMES, probs)):
    marker = ' ← predicted' if i == pred_class else ''
    print(f'  [{i:2d}] {cls:<15s} {p:.4f} ({p:.1%}){marker}')

---
## 6 — Explore More Samples

The cell below shows a batch of 20 random samples with their predictions, so you can get a broader sense of model performance.

In [ ]:
N_SAMPLES = 20
random_indices = np.random.choice(len(X_test), N_SAMPLES, replace=False)
X_batch = X_test[random_indices]
y_batch = y_test[random_indices]

batch_probs  = model.predict(X_batch, verbose=0)
batch_preds  = np.argmax(batch_probs, axis=1)
batch_conf   = np.max(batch_probs, axis=1)

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_batch[i].squeeze(), cmap='gray')
    correct = batch_preds[i] == y_batch[i]
    color   = '#27ae60' if correct else '#e74c3c'
    status  = '✓' if correct else '✗'
    ax.set_title(
        f'{status} {CLASS_NAMES[batch_preds[i]]}\nConf: {batch_conf[i]:.1%}\nTrue: {CLASS_NAMES[y_batch[i]]}',
        fontsize=8, color=color
    )
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(2.5)

correct_count = sum(batch_preds == y_batch)
plt.suptitle(
    f'Batch Predictions — {correct_count}/{N_SAMPLES} correct ({correct_count/N_SAMPLES:.0%})',
    fontsize=14, y=1.01
)
plt.tight_layout()
plt.show()

---
## About the Model

The **Tuned Mini-ResNet** is a residual convolutional network adapted for 28×28 grayscale Fashion MNIST images.

| Component | Detail |
|---|---|
| Architecture | Mini-ResNet (Functional API) |
| Initial conv | 64 filters, 3×3 |
| Stage 1 | 2 residual blocks @ 64 filters, 28×28 |
| Stage 2 | 2 residual blocks @ 128 filters, 14×14 |
| Stage 3 | 2 residual blocks @ 256 filters, 7×7 |
| Pooling | Global Average Pooling (replaces Flatten) |
| Output | Dense(10, softmax) |
| Optimizer | Adam + Cosine Decay |
| Regularization | L2 weight decay, Dropout, Data Augmentation, Batch Normalization |

**Key references:**
- He et al. (2016) — Deep Residual Learning for Image Recognition. CVPR.
- Xiao et al. (2017) — Fashion-MNIST: a Novel Image Dataset for Benchmarking ML Algorithms. arXiv:1708.07747.